## Lemmatisation preprocessing

This notebook writes **lemmatised plain text** next to the raw corpus. For each language-specific subtree under `data/` (`English_25_3`, `German_25_3`, `Russian_25_3`), it loads the matching **spaCy** pipeline, raises `nlp.max_length` for long novels, and replaces alphabetic tokens with their **lemma** while preserving original whitespace and non-alphabetic tokens (punctuation, numbers as produced by the tokenizer).

**Output:** mirror tree under `data/lemmas/<same subfolders>/`.

**Requirements:** `pip install spacy` and language models, e.g. `python -m spacy download en_core_web_sm de_core_news_sm ru_core_news_sm`.


In [ ]:
import os
from pathlib import Path

import spacy

# Raw texts live under data/<LangFolder>/ ; lemmatised copies go to data/lemmas/<LangFolder>/
INPUT_ROOT = Path("data")
OUTPUT_ROOT = Path("data") / "lemmas"

LANG_MODELS = {
    "English_25_3": "en_core_web_sm",
    "German_25_3": "de_core_news_sm",
    "Russian_25_3": "ru_core_news_sm",
}

nlp_cache = {}
for folder_name, model_name in LANG_MODELS.items():
    print(f"Loading {model_name} for {folder_name}...")
    nlp = spacy.load(model_name)
    # Default spaCy limit is low for full books
    nlp.max_length = 10_000_000
    nlp_cache[folder_name] = nlp

for root, _dirs, files in os.walk(INPUT_ROOT):
    root_path = Path(root)
    rel_root = root_path.relative_to(INPUT_ROOT)

    if len(rel_root.parts) == 0:
        continue

    top_folder = rel_root.parts[0]
    if top_folder not in nlp_cache:
        continue

    nlp = nlp_cache[top_folder]
    out_root = OUTPUT_ROOT / rel_root
    out_root.mkdir(parents=True, exist_ok=True)

    for fname in files:
        in_path = root_path / fname
        if not in_path.is_file():
            continue
        text = in_path.read_text(encoding="utf-8")
        doc = nlp(text)

        pieces = []
        for token in doc:
            if not token.is_punct and not token.is_space and token.is_alpha:
                pieces.append(token.lemma_ + token.whitespace_)
            else:
                pieces.append(token.text_with_ws)

        out_text = "".join(pieces)
        out_path = out_root / fname
        out_path.write_text(out_text, encoding="utf-8")
        print(f"{in_path} -> {out_path}")

print("Done.")
